# Titanic Survival Prediction – Data Cleaning & Preprocessing
### Coursework: Assignments 1-3 & Mini Project 1
**Dataset:** Titanic Dataset (Kaggle)

---

### Objectives & Checklist:
- **Assignment 1:** Load dataset using Pandas and summarize basic statistics (.info(), .describe()).
- **Assignment 2:** Handle missing data using mean and median imputation (Age, Embarked, Cabin).
- **Assignment 3:** Encode categorical variables using LabelEncoder & OneHotEncoder (Sex, Embarked).
- **Mini Project 1:**
  - Clean missing data.
  - Encode Sex and Embarked columns.
  - Visualize age distribution using Matplotlib & Seaborn.
  - Output cleaned dataset as a new CSV (	itanic_cleaned.csv).


## 1. Environment Setup & Library Imports
Import essential data manipulation, visualization, and preprocessing libraries.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Set visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 150


## 2. Assignment 1: Load Dataset & Summarize Basic Stats
Load the raw Titanic dataset using pd.read_csv() and inspect structure, types, and summary statistics.


In [ ]:
# Load dataset
df = pd.read_csv('titanic.csv')
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()


In [ ]:
# Inspect columns, data types, and non-null counts
df.info()


In [ ]:
# Numerical summary statistics (.describe())
df.describe().T


In [ ]:
# Categorical summary statistics
df.describe(include=['O']).T


In [ ]:
# Audit missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_summary = pd.DataFrame({'Missing_Count': missing, 'Percentage (%)': missing_pct})
missing_summary[missing_summary['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)


## 3. Assignment 2: Handle Missing Data Using Mean/Median Imputation

### Strategy Overview:
1. **Age (Numerical, ~19.9% missing):**
   - The Age distribution is right-skewed with a few elderly outliers.
   - **Mean:** $\approx 29.70$
   - **Median:** $\approx 28.00$
   - We compare Mean vs. Median imputation. The **Median** is preferred because it is robust against outliers and skewness.
2. **Embarked (Categorical, 2 missing):**
   - Imputed using the **Mode** ('S', Southampton).
3. **Cabin (Categorical, ~77.1% missing):**
   - More than three-quarters of values are missing. Direct imputation introduces artificial bias. We drop the column and create a binary feature Has_Cabin indicating cabin presence.


In [ ]:
# Statistical comparison of Age before and after Mean vs Median imputation
age_raw = df['Age']
age_mean = age_raw.mean()
age_median = age_raw.median()

print(f'Raw Age -> Mean: {age_mean:.2f}, Median: {age_median:.2f}, Std: {age_raw.std():.2f}, Skew: {age_raw.skew():.2f}')

# Imputation comparison
age_mean_imputed = age_raw.fillna(age_mean)
age_median_imputed = age_raw.fillna(age_median)

comp_df = pd.DataFrame({
    'Metric': ['Mean', 'Median', 'Std Dev', 'Min', 'Max'],
    'Raw (DropNA)': [age_raw.mean(), age_raw.median(), age_raw.std(), age_raw.min(), age_raw.max()],
    'Mean Imputed': [age_mean_imputed.mean(), age_mean_imputed.median(), age_mean_imputed.std(), age_mean_imputed.min(), age_mean_imputed.max()],
    'Median Imputed': [age_median_imputed.mean(), age_median_imputed.median(), age_median_imputed.std(), age_median_imputed.min(), age_median_imputed.max()]
})
comp_df


In [ ]:
# Execute Imputation on a working DataFrame
df_clean = df.copy()

# 1. Age Imputation via Scikit-Learn SimpleImputer (strategy='median')
imputer_median = SimpleImputer(strategy='median')
df_clean['Age'] = imputer_median.fit_transform(df_clean[['Age']])

# 2. Embarked Imputation via Mode ('S')
embarked_mode = df_clean['Embarked'].mode()[0]
df_clean['Embarked'].fillna(embarked_mode, inplace=True)

# 3. Cabin: Engineer 'Has_Cabin' flag and drop sparse 'Cabin' column
df_clean['Has_Cabin'] = df_clean['Cabin'].notnull().astype(int)
df_clean.drop(columns=['Cabin'], inplace=True)

print('Null counts after missing value handling:')
print(df_clean.isnull().sum())


## 4. Assignment 3: Encode Categorical Variables (LabelEncoder & OneHotEncoder)

### Concepts:
- **LabelEncoder:** Converts each unique category into an integer (, 1, 2, \dots$). Best suited for **binary** categories (e.g. Sex: female=0, male=1) or target variables ($). For nominal features with $>2$ categories (like Embarked), it can falsely imply an ordinal ranking.
- **OneHotEncoder:** Converts categorical columns into multiple binary (dummy) columns. Ideal for **nominal** multi-class variables (like Embarked $\rightarrow$ Southampton, Cherbourg, Queenstown) where no mathematical ordering exists.


In [ ]:
# 1. LabelEncoder on 'Sex'
le_sex = LabelEncoder()
df_clean['Sex'] = le_sex.fit_transform(df_clean['Sex'])
print('Sex LabelEncoder mapping:')
for cat, code in zip(le_sex.classes_, le_sex.transform(le_sex.classes_)):
    print(f'  {cat} -> {code}')

# Demonstration of LabelEncoder on Embarked (and discussion of limitations)
le_embarked = LabelEncoder()
embarked_le_demo = le_embarked.fit_transform(df_clean['Embarked'])
print('\nEmbarked LabelEncoder mapping (Demo):')
for cat, code in zip(le_embarked.classes_, le_embarked.transform(le_embarked.classes_)):
    print(f'  {cat} -> {code}')


In [ ]:
# 2. OneHotEncoder on 'Embarked'
ohe = OneHotEncoder(sparse_output=False, drop=None)
embarked_ohe = ohe.fit_transform(df_clean[['Embarked']])
embarked_cols = [f'Embarked_{c}' for c in ohe.categories_[0]]

embarked_df = pd.DataFrame(embarked_ohe, columns=embarked_cols, index=df_clean.index)

# Preview OneHotEncoded columns side-by-side with original
display_sample = pd.concat([df_clean[['Embarked']], embarked_df], axis=1).head(5)
print('OneHotEncoded Embarked feature columns:')
print(display_sample)

# Integrate one-hot columns into df_clean and drop original 'Embarked'
df_clean = pd.concat([df_clean, embarked_df], axis=1)
df_clean.drop(columns=['Embarked'], inplace=True)


## 5. Mini Project 1: Visualize Age Distribution (Matplotlib & Seaborn)
Visualizing age distributions to extract patterns and verify imputation impact:
1. **Histogram & KDE:** Raw Age vs Median-Imputed Age.
2. **KDE Plot:** Age Distribution by Survival Outcome (Survived = 0 vs 1).
3. **Boxplot & Stripplot:** Age Distribution across Socioeconomic Classes (Pclass 1, 2, 3).
4. **Split Violin Plot:** Age Distribution across Gender and Survival Status.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Titanic Survival Analysis: Age Distribution & Cleaning Visualizations', fontsize=18, fontweight='bold', y=0.98)

# Panel 1: Imputation Comparison
ax1 = axes[0, 0]
sns.histplot(df['Age'].dropna(), color='#1f77b4', kde=True, stat='density', label='Raw Age (Pre-Imputation)', ax=ax1, bins=30, alpha=0.5)
sns.histplot(df_clean['Age'], color='#ff7f0e', kde=True, stat='density', label=f'Imputed Age (Median = {age_median:.1f})', ax=ax1, bins=30, alpha=0.3)
ax1.axvline(age_median, color='red', linestyle='--', linewidth=2, label=f'Median: {age_median:.1f}')
ax1.axvline(age_mean, color='green', linestyle=':', linewidth=2, label=f'Mean: {age_mean:.1f}')
ax1.set_title('1. Age Distribution: Raw vs Median Imputed', fontsize=13, fontweight='bold')
ax1.set_xlabel('Age (Years)')
ax1.set_ylabel('Density')
ax1.legend(loc='upper right')

# Panel 2: Age Density by Survival Status
ax2 = axes[0, 1]
sns.kdeplot(data=df_clean[df_clean['Survived'] == 0]['Age'], label='Did Not Survive (0)', color='#d62728', fill=True, alpha=0.4, linewidth=2, ax=ax2)
sns.kdeplot(data=df_clean[df_clean['Survived'] == 1]['Age'], label='Survived (1)', color='#2ca02c', fill=True, alpha=0.4, linewidth=2, ax=ax2)
ax2.set_title('2. Age Density by Survival Status (KDE)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Age (Years)')
ax2.set_ylabel('Density')
ax2.legend(loc='upper right')

# Panel 3: Age by Passenger Class
ax3 = axes[1, 0]
sns.boxplot(x='Pclass', y='Age', data=df_clean, palette='Blues_r', ax=ax3, width=0.5, boxprops=dict(alpha=0.8), hue='Pclass', legend=False)
sns.stripplot(x='Pclass', y='Age', data=df_clean, color='black', alpha=0.2, jitter=0.2, size=3, ax=ax3)
ax3.set_title('3. Age Distribution Across Passenger Classes', fontsize=13, fontweight='bold')
ax3.set_xlabel('Passenger Class (1 = 1st, 2 = 2nd, 3 = 3rd)')
ax3.set_ylabel('Age (Years)')

# Panel 4: Age by Gender and Survival
ax4 = axes[1, 1]
df_plot = df_clean.copy()
df_plot['Gender'] = df_plot['Sex'].map({0: 'Female', 1: 'Male'})
df_plot['Survival_Status'] = df_plot['Survived'].map({0: 'Perished', 1: 'Survived'})
sns.violinplot(x='Gender', y='Age', hue='Survival_Status', data=df_plot, split=True, palette={'Perished': '#e74c3c', 'Survived': '#2ecc71'}, inner='quartile', ax=ax4)
ax4.set_title('4. Age Distribution by Gender and Survival (Split Violin)', fontsize=13, fontweight='bold')
ax4.set_xlabel('Gender')
ax4.set_ylabel('Age (Years)')
ax4.legend(title='Outcome', loc='upper right')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('age_distribution.png', bbox_inches='tight')
plt.show()


## 6. Output Cleaned Dataset as New CSV
Drop non-predictive string identifiers (PassengerId, Name, Ticket) to prepare the final feature set for machine learning models, then export to 	itanic_cleaned.csv.


In [ ]:
# Drop identifier columns unsuited for numerical models
ml_ready_df = df_clean.drop(columns=['PassengerId', 'Name', 'Ticket'])

# Export to CSV
ml_ready_df.to_csv('titanic_cleaned.csv', index=False)
print(f'Cleaned dataset saved successfully! Shape: {ml_ready_df.shape}')
ml_ready_df.head(10)


In [ ]:
# Final validation: Confirm zero missing values and clean datatypes
print('Final Missing Values Check:')
print(ml_ready_df.isnull().sum())
print('\nCleaned Feature Types:')
print(ml_ready_df.dtypes)
